# Donor-bulk recovery of single-cell signals

CLAMPfull is fitted to one raw-count-summed library per donor. Ground-truth donor composition is used only for LV matching and evaluation. The matched LVs are then projected into the original individual cells using the donor-bulk training statistics.


💡 **Environment:** `clamp-analyses`


In [ ]:
suppressPackageStartupMessages({
  library(data.table)
  library(ggplot2)
  library(cowplot)
  library(ggrastr)
  library(rhdf5)
})

DATASETS <- unlist(snakemake@params[["datasets"]])
PROD <- snakemake@params[["production_root"]]
OUT <- snakemake@params[["out_dir"]]
dir.create(OUT, recursive = TRUE, showWarnings = FALSE)
pretty_dataset <- c(Brain_Mathys2023="Brain: Mathys", Brain_Xiong2023="Brain: Xiong",
  Heart_Datar2026="Heart: Datar", PBMC_1k1k="PBMC: 1k1k",
  PBMC_Perez2022="PBMC: Perez", Lung_Sikkema2023="Lung: Sikkema")
dataset_colors <- c(Brain_Mathys2023="#5B8FF9", Brain_Xiong2023="#9270CA",
  Heart_Datar2026="#E8684A", PBMC_1k1k="#5AD8A6",
  PBMC_Perez2022="#F6BD16", Lung_Sikkema2023="#6DC8EC")


In [ ]:
assignments <- fread(snakemake@input[["assignments"]])
predictions <- fread(snakemake@input[["predictions"]])
metrics <- fread(snakemake@input[["metrics"]])
comparison <- fread(snakemake@input[["comparison"]])
recovery <- fread(snakemake@input[["recovery"]])
specificity <- fread(snakemake@input[["specificity"]])
specificity_summary <- fread(snakemake@input[["specificity_summary"]])
validation <- fread(snakemake@input[["validation"]])

valid_types <- metrics[valid_all_folds == TRUE, .(dataset, cell_type)]
fig2_predictions <- merge(predictions, valid_types, by=c("dataset", "cell_type"))
fig2_predictions[, dataset_label := pretty_dataset[dataset]]
fig2_statistics <- fig2_predictions[, {
  test <- if (.N > 2 && sd(observed) > 0 && sd(predicted) > 0) cor.test(observed, predicted) else NULL
  list(r=if (is.null(test)) NA_real_ else unname(test$estimate),
       p_value=if (is.null(test)) NA_real_ else test$p.value,
       n_predictions=.N, n_types=uniqueN(cell_type))
}, by=.(dataset, dataset_label)]
fwrite(fig2_predictions, snakemake@output[["fig2_predictions"]])
fwrite(fig2_statistics, snakemake@output[["fig2_statistics"]])
fig2_statistics


## Full-data and held-out donor recovery


In [ ]:
full_summary <- assignments[, .(n_types=.N, mean_r=mean(cor), median_r=median(cor)), by=dataset]
cv_summary <- metrics[valid_all_folds == TRUE, .(valid_types=.N, mean_oof_r=mean(pearson_r),
  mean_predictive_r2=mean(predictive_r2), mean_mae=mean(mae)), by=dataset]
merge(full_summary, cv_summary, by="dataset")


In [ ]:
ggplot(fig2_predictions, aes(observed, predicted, colour=dataset)) +
  rasterise(geom_point(size=.35, alpha=.35), dpi=180) +
  geom_smooth(method="lm", se=FALSE, linewidth=.45) +
  facet_wrap(~dataset_label, scales="free", ncol=3) +
  scale_colour_manual(values=dataset_colors) + theme_classic(base_size=8) +
  theme(legend.position="none") + labs(x="Observed cell fraction", y="OOF predicted fraction")


## Projection back to the original cells


In [ ]:
decode_h5 <- function(x) as.character(x)
read_feature_data <- function(dataset_id, selected_types=NULL) {
  umap_path <- file.path(PROD, dataset_id, "single_cell_umap", "umap_points.csv")
  h5_path <- file.path(PROD, dataset_id, "single_cell_projection", "single_cell_lv_scores.h5")
  xy <- fread(umap_path)
  chosen <- assignments[dataset == dataset_id]
  if (!is.null(selected_types)) chosen <- chosen[cell_type %in% selected_types]
  chosen <- chosen[order(-cor)]
  lv_names <- decode_h5(h5read(h5_path, "lv_names"))
  idx <- match(chosen$LV, lv_names)
  stopifnot(all(!is.na(idx)))
  # h5py writes cells x LVs, which rhdf5 exposes in reversed dimension order.
  # Sequentially read the selected LV rows, then subset sampled cell columns
  # in memory; irregular HDF5 point selection is prohibitively slow here.
  score_rows <- h5read(h5_path, "scores", index=list(idx, NULL), drop=FALSE)
  scores <- t(score_rows[, xy$cell_index + 1L, drop=FALSE])
  rm(score_rows)
  activity <- rbindlist(lapply(seq_len(nrow(chosen)), function(j) {
    values <- as.numeric(scores[,j]); limits <- quantile(values, c(.01,.99), na.rm=TRUE)
    values <- pmin(pmax(values, limits[1]), limits[2])
    data.table(dataset=dataset_id, cell_index=xy$cell_index, cell_id=xy$cell_id,
      umap1=xy$umap1, umap2=xy$umap2, cell_type=chosen$cell_type[j],
      LV=chosen$LV[j], assignment_r=chosen$cor[j], activity=values)
  }))
  list(annotation=xy, activity=activity)
}

perez_types <- c("B_cell", "Myeloid", "NK", "T_cell")
perez <- read_feature_data("PBMC_Perez2022", perez_types)
perez_annotation <- perez$annotation[mapped_cell_type %in% perez_types]
perez_activity <- perez$activity[cell_index %in% perez_annotation$cell_index]
perez_recovery <- recovery[dataset == "PBMC_Perez2022",
  .(cell_type, recovery_pct, purity_lift)]
perez_activity <- merge(perez_activity, perez_recovery, by = "cell_type",
  all.x = TRUE, sort = FALSE)
stopifnot(!anyNA(perez_activity$recovery_pct))
fwrite(perez_annotation, snakemake@output[["perez_annotation"]])
fwrite(perez_activity, snakemake@output[["perez_activity"]])
perez_summary <- data.table(
  dataset="PBMC_Perez2022",
  median_auroc=specificity[dataset=="PBMC_Perez2022" & is_match==TRUE, median(auc)],
  median_bulk_r=assignments[dataset=="PBMC_Perez2022", median(cor)])
fwrite(perez_summary, snakemake@output[["perez_summary"]])

# Compact, data-only exports for Supplementary Figure 2. Each cohort keeps
# the four strongest bulk proportion assignments; activity columns share
# the fixed expression-derived UMAP and are never used to recompute it.
supp2_lvs <- assignments[order(dataset, -cor), head(.SD, 4L), by=dataset]
supp2_lvs[, plot_order := seq_len(.N), by=dataset]
supp2_lvs <- merge(
  supp2_lvs,
  recovery[, .(dataset, cell_type, recovery_pct, purity_lift)],
  by=c("dataset", "cell_type"), all.x=TRUE, sort=FALSE)
supp2_lvs <- merge(
  supp2_lvs,
  specificity[is_match==TRUE,
    .(dataset, cell_type=assigned_cell_type, auroc=auc)],
  by=c("dataset", "cell_type"), all.x=TRUE, sort=FALSE)
supp2_lvs[, dataset_order := match(dataset, DATASETS)]
setorder(supp2_lvs, dataset_order, plot_order)
supp2_lvs[, dataset_order := NULL]
stopifnot(supp2_lvs[, .N, by=dataset][, all(N == 4L)],
          !anyNA(supp2_lvs[, .(LV, cor, recovery_pct, auroc)]))

supp2_cells <- rbindlist(lapply(DATASETS, function(dataset_id) {
  selected <- supp2_lvs[dataset == dataset_id][order(plot_order)]
  dat <- read_feature_data(dataset_id, selected$cell_type)
  annotation <- dat$annotation[, .(dataset=dataset_id, cell_index, umap1, umap2,
                                    mapped_cell_type)]
  long <- merge(
    dat$activity[, .(dataset, cell_index, cell_type, activity)],
    selected[, .(dataset, cell_type, plot_order)],
    by=c("dataset", "cell_type"), all.x=TRUE, sort=FALSE)
  stopifnot(!anyNA(long$plot_order))
  long[, activity_name := paste0("activity_", plot_order)]
  wide <- dcast(long, dataset + cell_index ~ activity_name, value.var="activity")
  out <- merge(annotation, wide, by=c("dataset", "cell_index"),
               all.x=TRUE, sort=FALSE)
  setorder(out, cell_index)
  stopifnot(nrow(out) == nrow(annotation),
            !anyNA(out[, paste0("activity_", 1:4), with=FALSE]))
  out
}))
stopifnot(supp2_cells[, uniqueN(cell_index), by=dataset][,
                       all(V1 == 60000L)])
supp2_purity <- recovery[, .(dataset, cell_type, recovery_pct)]
fwrite(supp2_purity, snakemake@output[["supp2_purity"]])
fwrite(supp2_cells, snakemake@output[["supp2_umap_cells"]])
fwrite(supp2_lvs[, .(dataset, plot_order, cell_type, LV, assignment_r=cor,
                      recovery_pct, purity_lift, auroc)],
       snakemake@output[["supp2_umap_lvs"]])
perez_summary


## Single-cell localization summary


In [ ]:
ggplot(recovery, aes(reorder(pretty_dataset[dataset], recovery_pct, median), recovery_pct, fill=dataset)) +
  geom_boxplot(outlier.shape=NA, width=.6, linewidth=.3) + geom_jitter(width=.12, size=.55, alpha=.6) +
  scale_fill_manual(values=dataset_colors) + theme_classic(base_size=8) +
  theme(legend.position="none", axis.text.x=element_text(angle=25,hjust=1)) +
  labs(x=NULL, y="Top-1% annotated-cell purity (%)")


In [ ]:
ggplot(specificity, aes(observed_cell_type, assigned_cell_type, fill=auc)) +
  geom_tile(colour="white", linewidth=.15) +
  facet_wrap(~pretty_dataset[dataset], scales="free", ncol=3) +
  scale_fill_gradientn(colours=c("#2166AC","white","#B2182B"), limits=c(0,1)) +
  theme_minimal(base_size=6) + theme(panel.grid=element_blank(), axis.text.x=element_text(angle=45,hjust=1)) +
  labs(x="Annotated cell type", y="Bulk-matched LV", fill="AUROC")


## Expression UMAP FeaturePlot sheets for all cohorts

Every sheet uses one fixed expression-derived UMAP. The annotation and all matched LV activities are overlays on those same coordinates; no LV-derived embedding is computed.


In [ ]:
cell_palette <- c(B_cell="#4477AA", Myeloid="#EE6677", NK="#228833", T_cell="#CCBB44",
  CD4_T="#66CCEE", CD8_T="#AA3377", CD14_Mono="#EE7733", CD16_Mono="#0077BB",
  DC="#BBBBBB", Plasma_B="#EE3377", gd_T="#009988")
feature_sheet <- function(dataset_id) {
  dat <- read_feature_data(dataset_id)
  labels <- sort(unique(dat$annotation$mapped_cell_type))
  missing <- setdiff(labels, names(cell_palette))
  palette <- cell_palette
  if (length(missing)) palette <- c(palette, setNames(grDevices::hcl.colors(length(missing), "Dark 3"), missing))
  ann <- ggplot(dat$annotation, aes(umap1,umap2,colour=mapped_cell_type)) +
    rasterise(geom_point(size=.18, alpha=.8), dpi=170) + scale_colour_manual(values=palette) +
    coord_equal() + theme_void(base_size=7) + theme(legend.position="bottom") + ggtitle(pretty_dataset[dataset_id])
  plots <- lapply(unique(dat$activity$cell_type), function(ct) {
    d <- dat$activity[cell_type==ct]
    high <- if (ct %in% names(palette)) palette[[ct]] else "#B2182B"
    ggplot(d, aes(umap1, umap2, colour=activity)) +
      rasterise(geom_point(size=.18), dpi=170) +
      scale_colour_gradientn(colours=c("#D8E2EF","white",high)) + coord_equal() + theme_void(base_size=7) +
      theme(legend.position="none", panel.border=element_rect(colour="black",fill=NA,linewidth=.25)) +
      ggtitle(paste0(ct, " · ", unique(d$LV), "  r=", sprintf("%.2f",unique(d$assignment_r))))
  })
  plot_grid(plotlist=c(list(ann),plots), ncol=4)
}
for (dataset_id in DATASETS) {
  options(repr.plot.width=12, repr.plot.height=max(4, ceiling((1+nrow(assignments[dataset==dataset_id]))/4)*3))
  print(feature_sheet(dataset_id))
}


Only CSV files are exported. Supplemental candidates remain inline in the executed notebook; the publication panel notebook is the sole writer of PNG, PDF, and SVG files.
